# Forecast Metrics Introduction

This notebook introduces the most commonly used forecast evaluation metrics.
Understanding these metrics is essential for assessing how well your forecasting models perform.

**Topics covered:**
- **MAE** - Mean Absolute Error
- **RMSE** - Root Mean Square Error
- **MAPE** - Mean Absolute Percentage Error
- **MASE** - Mean Absolute Scaled Error

We will use Brazilian macroeconomic data to illustrate each metric with practical examples.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from forecastbox.metrics import mae, rmse, mape, mase

import sys
sys.path.insert(0, "..")
from utils.helpers import load_macro_brazil, load_macro_us, plot_series

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 4)

## 1. Loading Data

We use two macroeconomic datasets generated for this tutorial:

- **macro_brazil.csv**: Brazilian macro indicators (GDP growth, inflation, interest rate, unemployment, exchange rate)
- **macro_us.csv**: US macro indicators (GDP growth, CPI inflation, Fed Funds rate, unemployment)

Let's start by loading and exploring the Brazilian dataset.

In [ ]:
df_brazil = load_macro_brazil()
print("Shape:", df_brazil.shape)
print()
df_brazil.head()

In [ ]:
df_brazil.info()

In [ ]:
# Create a simple "forecast" by shifting gdp_growth forward by 1 period (naive forecast)
gdp = df_brazil["gdp_growth"].dropna()
actual = gdp.iloc[1:].values
predicted = gdp.iloc[:-1].values  # naive: forecast = previous value
training = gdp.iloc[:60].values   # first 60 obs for MASE scaling

print(f"Evaluation period: {len(actual)} observations")
print(f"Actual (first 5):    {actual[:5].round(4)}")
print(f"Predicted (first 5): {predicted[:5].round(4)}")

## 2. MAE - Mean Absolute Error

$$MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$

**Interpretation:**
- MAE measures the average magnitude of errors, ignoring their direction.
- It is in the **same units** as the original data, making it easy to interpret.
- MAE treats all errors equally (unlike RMSE which penalizes large errors more).
- A MAE of 0.5 for GDP growth means the forecast is off by 0.5 percentage points on average.

In [ ]:
# Manual calculation
mae_manual = np.mean(np.abs(actual - predicted))
print(f"MAE (manual):      {mae_manual:.4f}")

# Using forecastbox
mae_fb = mae(actual, predicted)
print(f"MAE (forecastbox): {mae_fb:.4f}")

# Verify they match
assert np.isclose(mae_manual, mae_fb), "Values should match!"
print("\nInterpretation: the naive forecast is off by", f"{mae_fb:.4f}",
      "percentage points on average.")

## 3. RMSE - Root Mean Square Error

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

**Interpretation:**
- RMSE also measures error magnitude in the same units as the data.
- It **penalizes large errors more** than MAE due to squaring.
- RMSE >= MAE always. When RMSE >> MAE, it indicates the presence of large outlier errors.
- Preferred when large errors are particularly costly (e.g., financial risk).

In [ ]:
# Manual calculation
rmse_manual = np.sqrt(np.mean((actual - predicted) ** 2))
print(f"RMSE (manual):      {rmse_manual:.4f}")

# Using forecastbox
rmse_fb = rmse(actual, predicted)
print(f"RMSE (forecastbox): {rmse_fb:.4f}")

assert np.isclose(rmse_manual, rmse_fb)

# Compare with MAE
print(f"\nMAE:  {mae_fb:.4f}")
print(f"RMSE: {rmse_fb:.4f}")
print(f"RMSE/MAE ratio: {rmse_fb / mae_fb:.2f}")
print("A ratio close to 1.0 means errors are uniform; higher means some large errors exist.")

## 4. MAPE - Mean Absolute Percentage Error

$$MAPE = \frac{100}{n}\sum_{i=1}^{n}\left|\frac{y_i - \hat{y}_i}{y_i}\right|$$

**Interpretation:**
- MAPE expresses error as a **percentage** of the actual value.
- Scale-independent: useful for comparing forecasts across different series.

**Limitations:**
- **Undefined when actual values are zero** (division by zero).
- **Asymmetric**: penalizes over-forecasting more than under-forecasting.
- Can be extremely large when actual values are close to zero.
- Not suitable for data that crosses zero (e.g., growth rates that can be negative).

In [ ]:
# MAPE for inflation (positive values, MAPE works well)
inflation = df_brazil["inflation"].dropna()
actual_inf = inflation.iloc[1:].values
predicted_inf = inflation.iloc[:-1].values

mape_inf = mape(actual_inf, predicted_inf)
print(f"MAPE for inflation (naive): {mape_inf:.2f}%")

# MAPE for GDP growth (can have values near zero - problematic!)
mape_gdp = mape(actual, predicted)
print(f"MAPE for GDP growth (naive): {mape_gdp:.2f}%")
print("\nNote: MAPE can be very high or infinite when actual values are near zero.")

## 5. MASE - Mean Absolute Scaled Error

$$MASE = \frac{\frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|}{\frac{1}{T-1}\sum_{t=2}^{T}|y_t - y_{t-1}|}$$

**Interpretation:**
- MASE scales the forecast error by the in-sample MAE of a **naive forecast**.
- **MASE < 1**: the model outperforms the naive baseline.
- **MASE = 1**: the model performs the same as the naive baseline.
- **MASE > 1**: the model is worse than the naive baseline.

**Advantages over MAPE:**
- Well-defined even when actual values are zero.
- Symmetric (no bias towards over- or under-forecasting).
- Scale-independent: can compare across different series.
- Recommended by Hyndman & Koehler (2006) as a general-purpose metric.

In [ ]:
# MASE requires the training series for scaling
# Split: first 80% for training, last 20% for evaluation
n = len(gdp)
split = int(0.8 * n)

train_series = gdp.iloc[:split].values
test_actual = gdp.iloc[split:].values
test_naive = gdp.iloc[split - 1 : -1].values  # naive forecast for test period

mase_value = mase(test_actual, test_naive, training_series=train_series)
print(f"MASE (naive forecast): {mase_value:.4f}")
print("\nBy definition, the naive forecast should have MASE close to 1.0")
print("(not exactly 1.0 because train and test MAE differ).")

# A "better" forecast: average of last 3 values
test_sma3 = pd.Series(gdp.values).rolling(3).mean().dropna().iloc[split - 3:].values[:len(test_actual)]
mase_sma3 = mase(test_actual, test_sma3, training_series=train_series)
print(f"\nMASE (SMA-3 forecast): {mase_sma3:.4f}")
if mase_sma3 < 1.0:
    print("SMA-3 beats the naive baseline!")
else:
    print("SMA-3 does not beat the naive baseline.")

## 6. Comparing Metrics

Each metric has strengths and weaknesses. Here is a guide for choosing:

| Metric | Scale-dependent? | Handles zeros? | Penalizes large errors? | Best for |
|--------|:---:|:---:|:---:|---|
| MAE | Yes | Yes | Equally | General-purpose, interpretable |
| RMSE | Yes | Yes | More heavily | When large errors are costly |
| MAPE | No (%) | **No** | Equally | Comparing across scales (positive data) |
| MASE | No (scaled) | Yes | Equally | Comparing across series, benchmarking |

In [ ]:
# Compare metrics across different forecast methods for GDP growth
n = len(gdp)
split = int(0.8 * n)
train = gdp.iloc[:split]
test = gdp.iloc[split:]

# Different forecasting approaches
forecasts = {
    "Naive (last value)": np.full(len(test), train.iloc[-1]),
    "Mean": np.full(len(test), train.mean()),
    "SMA-6": pd.Series(gdp.values).rolling(6).mean().dropna().values[split - 1 : split - 1 + len(test)],
    "SMA-12": pd.Series(gdp.values).rolling(12).mean().dropna().values[split - 1 : split - 1 + len(test)],
}

# Build comparison table
rows = []
for name, pred in forecasts.items():
    pred = pred[:len(test)]  # ensure same length
    rows.append({
        "Method": name,
        "MAE": mae(test.values, pred),
        "RMSE": rmse(test.values, pred),
        "MAPE": mape(test.values, pred),
        "MASE": mase(test.values, pred, training_series=train.values),
    })

comparison = pd.DataFrame(rows).set_index("Method")
print("Metric Comparison for GDP Growth Forecasts")
print("=" * 60)
comparison.round(4)

## Exercise 1: Calculate all metrics for US data

Load the US macroeconomic dataset (`macro_us.csv`) and compute MAE, RMSE, MAPE, and MASE
for a naive forecast of `gdp_growth`. Use an 80/20 train/test split.

In [ ]:
# TODO: Exercise 1 - Load macro_us.csv and compute MAE, RMSE, MAPE, MASE for gdp_growth

## Exercise 2: Which metric is most appropriate for inflation forecasting? Why?

Consider the properties of inflation data (always positive, can be close to zero in low-inflation
environments, sometimes has large spikes). Load both datasets and experiment with different metrics
to support your answer.

In [ ]:
# TODO: Exercise 2 - Your analysis here